# Day 6 — Course Recommender TF-IDF Pipeline
## Notebook: `04_courses_pipeline.ipynb`

**Goal:** Build the sklearn Pipeline that turns the Coursera CSV into 3 trained  
pickles ready for the FastAPI backend.

**Algorithm:** TF-IDF vectorisation + cosine similarity  
(content-based filtering — no user ratings needed)

**Pipeline stages:**  
`CourseLoader` → `CourseCleaner` → `TFIDFTransformer` → `CourseSimilarityBuilder`

**Output pickles** (saved to `../models/`):  
- `course_tfidf.pkl`  — fitted TfidfVectorizer  
- `course_sim.pkl`    — cosine similarity matrix  
- `courses_df.pkl`    — full course DataFrame with combined_text  

> Run all cells top to bottom. `../data/coursera_courses.csv` must exist.

In [11]:
import pandas as pd
import numpy as np
import pickle
import warnings
import os
warnings.filterwarnings('ignore')

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

os.makedirs('../models', exist_ok=True)
print("Imports successful")

Imports successful


## 1. Custom transformer: CourseLoader

In [12]:
class CourseLoader(BaseEstimator, TransformerMixin):
    """
    Stage 1: Load the Coursera CSV and standardise column names.

    Different Kaggle versions of the dataset use different column names.
    This transformer detects whichever naming convention is present
    and maps everything to a standard set:
      course_title, subject, level, rating, course_url

    Input  : None
    Output : dict with key 'raw_df'
    """
    def __init__(self, csv_path='../data/coursera_courses.csv'):
        self.csv_path = csv_path

    def fit(self, X=None, y=None):
        return self

    def transform(self, X=None):
        df = pd.read_csv(self.csv_path)
        print(f"[CourseLoader]  loaded {df.shape[0]:,} rows  x  {df.shape[1]} columns")
        print(f"[CourseLoader]  original columns: {df.columns.tolist()}")

        col_map = {}

        # Detect title column
        for c in ['Course Name', 'course_title', 'name', 'title', 'Course Title', 'CourseName']:
            if c in df.columns:
                col_map[c] = 'course_title'
                break

        # Detect subject / category
        for c in ['Course Description', 'subject', 'category', 'Subject',
                  'skills', 'Category', 'Description']:
            if c in df.columns:
                col_map[c] = 'subject'
                break

        # Detect level / difficulty
        for c in ['Difficulty Level', 'level', 'difficulty', 'Level', 'DifficultyLevel']:
            if c in df.columns:
                col_map[c] = 'level'
                break

        # Detect rating
        for c in ['Course Rating', 'rating', 'Rating', 'score', 'CourseRating']:
            if c in df.columns:
                col_map[c] = 'rating'
                break

        # Detect URL
        for c in ['Course URL', 'course_url', 'url', 'URL', 'link', 'CourseURL']:
            if c in df.columns:
                col_map[c] = 'course_url'
                break

        df = df.rename(columns=col_map)
        return {'raw_df': df}

print("CourseLoader defined")

CourseLoader defined


## 2. Custom transformer: CourseCleaner

In [13]:
class CourseCleaner(BaseEstimator, TransformerMixin):
    """
    Stage 2: Clean and standardise the raw course DataFrame.

    - Select only the 5 columns we need
    - Drop rows with missing course_title (critical field)
    - Fill missing subject, level with sensible defaults
    - Coerce rating to float, fill missing with median
    - Drop duplicate titles
    - Standardise level strings to title case
    - Build combined_text = title + subject + level (TF-IDF input)

    Input  : data dict with 'raw_df'
    Output : data dict with 'courses_df' added
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        data = X.copy()
        df   = data['raw_df'].copy()

        # Keep only columns we need
        keep = [c for c in ['course_title','subject','level','rating','course_url']
                if c in df.columns]
        df = df[keep].copy()

        # Add missing columns with defaults
        if 'subject'    not in df.columns: df['subject']    = 'General'
        if 'level'      not in df.columns: df['level']      = 'Mixed'
        if 'rating'     not in df.columns: df['rating']     = 4.0
        if 'course_url' not in df.columns: df['course_url'] = ''

        # Drop rows missing critical field
        df.dropna(subset=['course_title'], inplace=True)

        # Fill missing values
        df['subject']    = df['subject'].fillna('General')
        df['level']      = df['level'].fillna('Mixed')
        df['course_url'] = df['course_url'].fillna('')

        # Coerce rating to numeric, fill missing with median
        df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
        df['rating'] = df['rating'].fillna(df['rating'].median())
        df['rating'] = df['rating'].clip(0, 5)

        # Standardise level to Title Case
        df['level'] = df['level'].str.strip().str.title()

        # Drop duplicate titles (keep first occurrence)
        df.drop_duplicates(subset=['course_title'], keep='first', inplace=True)
        df.reset_index(drop=True, inplace=True)

        # Build combined_text — this is what TF-IDF trains on
        df['combined_text'] = (
            df['course_title'].str.lower().fillna('') + ' ' +
            df['subject'].str.lower().fillna('')      + ' ' +
            df['level'].str.lower().fillna('')
        ).str.strip()

        data['courses_df'] = df
        print(f"[CourseCleaner]  cleaned DataFrame: {df.shape}")
        print(f"[CourseCleaner]  levels found: {sorted(df['level'].unique())}")
        print(f"[CourseCleaner]  rating range: {df['rating'].min():.1f} – {df['rating'].max():.1f}")
        return data

print("CourseCleaner defined")

CourseCleaner defined


## 3. Custom transformer: TFIDFTransformer

In [14]:
class TFIDFTransformer(BaseEstimator, TransformerMixin):
    """
    Stage 3: Fit a TF-IDF vectoriser on the combined_text column.

    TF-IDF (Term Frequency–Inverse Document Frequency):
      - TF  : how often a word appears in THIS course's text
      - IDF : penalises words that appear in MANY courses (common words are less informative)
      - bigrams (ngram_range=(1,2)) capture phrases like 'machine learning', 'data science'
      - max_features=5000 limits vocabulary size for speed

    At inference time:
      query_vec = vectorizer.transform(["machine learning"])
      scores    = cosine_similarity(query_vec, tf_matrix).flatten()

    Input  : data dict with 'courses_df'
    Output : data dict with 'vectorizer' and 'tf_matrix' added
    """
    def __init__(self, max_features=5000, ngram_range=(1, 2)):
        self.max_features = max_features
        self.ngram_range  = ngram_range

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        data = X.copy()
        df   = data['courses_df']

        vectorizer = TfidfVectorizer(
            max_features = self.max_features,
            ngram_range  = self.ngram_range,
            stop_words   = 'english',
            min_df       = 1,           # include rare terms (small dataset)
            sublinear_tf = True,        # apply log(1+tf) dampening
        )

        tf_matrix = vectorizer.fit_transform(df['combined_text'])

        data['vectorizer'] = vectorizer
        data['tf_matrix']  = tf_matrix

        vocab_size = len(vectorizer.vocabulary_)
        print(f"[TFIDFTransformer]  matrix shape : {tf_matrix.shape}")
        print(f"[TFIDFTransformer]  vocab size   : {vocab_size:,} features")
        print(f"[TFIDFTransformer]  ngram range  : {self.ngram_range}")

        # Show top-10 terms by IDF weight (most distinctive terms)
        idf_scores = dict(zip(vectorizer.get_feature_names_out(),
                              vectorizer.idf_))
        top_terms = sorted(idf_scores.items(), key=lambda x: x[1], reverse=True)[:10]
        print(f"[TFIDFTransformer]  top distinctive terms: {[t for t,_ in top_terms]}")

        return data

print("TFIDFTransformer defined")

TFIDFTransformer defined


## 4. Custom transformer: CourseSimilarityBuilder

In [15]:
class CourseSimilarityBuilder(BaseEstimator, TransformerMixin):
    """
    Stage 4: Pre-compute the full cosine similarity matrix.

    Shape: (n_courses, n_courses)
    sim_matrix[i, j] = cosine similarity between course i and course j

    Pre-computing is a memory-for-speed tradeoff:
      - Pro: near-instant query responses (just index a row)
      - Con: ~50MB storage for 3500 courses
      - Alternative: compute at query time (~30ms overhead per request)

    For a dataset of <5000 courses, pre-computing is the right choice.

    Input  : data dict with 'tf_matrix'
    Output : data dict with 'sim_matrix' added
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        data      = X.copy()
        tf_matrix = data['tf_matrix']

        # Dense cosine similarity matrix
        sim_matrix = cosine_similarity(tf_matrix, tf_matrix)

        data['sim_matrix'] = sim_matrix
        print(f"[CourseSimilarityBuilder]  similarity matrix: {sim_matrix.shape}")
        print(f"[CourseSimilarityBuilder]  matrix size: "
              f"{sim_matrix.nbytes / (1024**2):.1f} MB in memory")
        return data

print("CourseSimilarityBuilder defined")

CourseSimilarityBuilder defined


## 5. Assemble and run the pipeline

In [16]:
course_pipeline = Pipeline(steps=[
    ('loader',    CourseLoader()),
    ('cleaner',   CourseCleaner()),
    ('tfidf',     TFIDFTransformer(max_features=5000, ngram_range=(1, 2))),
    ('similarity', CourseSimilarityBuilder()),
])

print("Course pipeline stages:")
for name, step in course_pipeline.steps:
    print(f"  {name:12s} → {step.__class__.__name__}")
print()

output = course_pipeline.fit_transform(None)
print("\nPipeline complete. Keys:", list(output.keys()))

Course pipeline stages:
  loader       → CourseLoader
  cleaner      → CourseCleaner
  tfidf        → TFIDFTransformer
  similarity   → CourseSimilarityBuilder

[CourseLoader]  loaded 3,522 rows  x  7 columns
[CourseLoader]  original columns: ['Course Name', 'University', 'Difficulty Level', 'Course Rating', 'Course URL', 'Course Description', 'Skills']
[CourseCleaner]  cleaned DataFrame: (3416, 6)
[CourseCleaner]  levels found: ['Advanced', 'Beginner', 'Conversant', 'Intermediate', 'Not Calibrated']
[CourseCleaner]  rating range: 1.0 – 5.0
[TFIDFTransformer]  matrix shape : (3416, 5000)
[TFIDFTransformer]  vocab size   : 5,000 features
[TFIDFTransformer]  ngram range  : (1, 2)
[TFIDFTransformer]  top distinctive terms: ['ciudades', 'gbv', 'process mining', 'villes', 'aboriginal', 'arbitration', 'blues', 'dairy', 'doping', 'evs']
[CourseSimilarityBuilder]  similarity matrix: (3416, 3416)
[CourseSimilarityBuilder]  matrix size: 89.0 MB in memory

Pipeline complete. Keys: ['raw_df', 'cou

## 6. Recommendation function — test before pickling

In [17]:
def recommend_courses(topic: str, level: str = None,
                      branch: str = None, n: int = 10,
                      courses_df=None, vectorizer=None, tf_matrix=None):
    """
    Return top-n course recommendations for a topic query.

    Parameters
    ----------
    topic      : str   query string (e.g. 'machine learning', 'python basics')
    level      : str   optional filter — 'Beginner', 'Intermediate', 'Advanced'
    branch     : str   optional — 'CSE' applies 1.2x boost to CS/ML courses
    n          : int   number of results (default 10)
    
    Returns
    -------
    list of dicts with keys: course_title, subject, level, rating, course_url, score
    """
    if courses_df  is None: courses_df  = output['courses_df']
    if vectorizer  is None: vectorizer  = output['vectorizer']
    if tf_matrix   is None: tf_matrix   = output['tf_matrix']

    # Transform query using fitted vectorizer
    query_vec = vectorizer.transform([topic.lower()])

    # Compute cosine similarity between query and all courses
    scores = cosine_similarity(query_vec, tf_matrix).flatten()

    result_df = courses_df.copy()
    result_df['score'] = scores

    # Apply level filter if specified
    if level:
        result_df = result_df[
            result_df['level'].str.lower() == level.lower()
        ]
        if len(result_df) == 0:
            # Fall back to all levels if filter leaves nothing
            result_df = courses_df.copy()
            result_df['score'] = scores

    # Apply branch boost (CSE users get CS/ML/Data courses ranked higher)
    if branch and branch.upper() == 'CSE':
        cs_keywords = r'computer|data|machine|python|programming|software|algorithm|network'
        cs_mask = result_df['subject'].str.contains(cs_keywords, case=False, na=False)
        result_df.loc[cs_mask, 'score'] *= 1.2

    # Sort by score and take top-n
    top = result_df.nlargest(n, 'score')

    results = []
    for _, row in top.iterrows():
        results.append({
            'course_title': row['course_title'],
            'subject'     : row['subject'],
            'level'       : row['level'],
            'rating'      : round(float(row['rating']), 2),
            'course_url'  : row['course_url'],
            'score'       : round(float(row['score']), 4),
        })
    return results


# ── Test with 5 different queries ──────────────────────────────────
test_queries = [
    ('machine learning', None, None),
    ('python programming', 'Beginner', None),
    ('data science', None, 'CSE'),
    ('web development', 'Intermediate', None),
    ('algorithms', None, 'CSE'),
]

for topic, level, branch in test_queries:
    results = recommend_courses(topic, level=level, branch=branch, n=5)
    print(f"\nQuery: '{topic}'  level={level}  branch={branch}")
    for i, r in enumerate(results[:3], 1):
        print(f"  {i}. [{r['score']:.3f}] {r['course_title'][:55]}  ({r['level']})")


Query: 'machine learning'  level=None  branch=None
  1. [0.454] Introduction to Applied Machine Learning  (Intermediate)
  2. [0.404] Machine Learning for Data Analysis  (Intermediate)
  3. [0.387] How Google does Machine Learning  (Beginner)

Query: 'python programming'  level=Beginner  branch=None
  1. [0.437] Python Basics  (Beginner)
  2. [0.338] Python Programming Essentials  (Beginner)
  3. [0.335] Compose and Program Music in Python using Earsketch  (Beginner)

Query: 'data science'  level=None  branch=CSE
  1. [0.647] What is Data Science?  (Conversant)
  2. [0.581] Genomic Data Science with Galaxy  (Beginner)
  3. [0.579] Statistics for Genomic Data Science  (Intermediate)

Query: 'web development'  level=Intermediate  branch=None
  1. [0.167] Front-End Web Development with React  (Intermediate)
  2. [0.109] Ruby on Rails: An Introduction  (Intermediate)
  3. [0.104] Front-End Web UI Frameworks and Tools: Bootstrap 4  (Intermediate)

Query: 'algorithms'  level=None  branch=CS

## 7. Hybrid boost verification

In [18]:
# Show that CSE branch boost changes the ranking for CS-related queries
print("=== Branch boost effect for 'data science' query ===")
print()

results_no_boost = recommend_courses('data science', branch=None, n=5)
results_boosted  = recommend_courses('data science', branch='CSE', n=5)

print("Without branch boost:")
for i, r in enumerate(results_no_boost[:5], 1):
    print(f"  {i}. [{r['score']:.3f}]  {r['course_title'][:50]}")

print("\nWith CSE branch boost:")
for i, r in enumerate(results_boosted[:5], 1):
    print(f"  {i}. [{r['score']:.3f}]  {r['course_title'][:50]}")

=== Branch boost effect for 'data science' query ===

Without branch boost:
  1. [0.539]  What is Data Science?
  2. [0.484]  Genomic Data Science with Galaxy
  3. [0.482]  Statistics for Genomic Data Science
  4. [0.408]  A Crash Course in Data Science
  5. [0.391]  Python for Genomic Data Science

With CSE branch boost:
  1. [0.647]  What is Data Science?
  2. [0.581]  Genomic Data Science with Galaxy
  3. [0.579]  Statistics for Genomic Data Science
  4. [0.489]  A Crash Course in Data Science
  5. [0.469]  Python for Genomic Data Science


## 8. Export 3 pickles to models/

In [19]:
artifacts = {
    '../models/course_tfidf.pkl' : output['vectorizer'],
    '../models/course_sim.pkl'   : output['sim_matrix'],
    '../models/courses_df.pkl'   : output['courses_df'],
}

for path, obj in artifacts.items():
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"  Saved: {path:42s}  ({size_mb:.1f} MB)")

print("\nAll 3 course pickles saved to ../models/")

  Saved: ../models/course_tfidf.pkl                  (0.2 MB)
  Saved: ../models/course_sim.pkl                    (89.0 MB)
  Saved: ../models/courses_df.pkl                    (8.2 MB)

All 3 course pickles saved to ../models/


## 9. Verify pickles round-trip correctly

In [20]:
with open('../models/course_tfidf.pkl', 'rb') as f: chk_vec = pickle.load(f)
with open('../models/course_sim.pkl',   'rb') as f: chk_sim = pickle.load(f)
with open('../models/courses_df.pkl',   'rb') as f: chk_df  = pickle.load(f)

print("Pickle verification:")
print(f"  course_tfidf.pkl : {type(chk_vec).__name__}  vocab={len(chk_vec.vocabulary_):,}")
print(f"  course_sim.pkl   : ndarray {chk_sim.shape}")
print(f"  courses_df.pkl   : DataFrame {chk_df.shape}")
print(f"  columns          : {chk_df.columns.tolist()}")

# End-to-end test from pickled vectorizer (not precomputed sim matrix)
tf_from_pickle = chk_vec.transform(chk_df['combined_text'])
test_query     = chk_vec.transform(['machine learning python'])
scores         = cosine_similarity(test_query, tf_from_pickle).flatten()

chk_df_copy = chk_df.copy()
chk_df_copy['score'] = scores
top5 = chk_df_copy.nlargest(5, 'score')

print("\nEnd-to-end test for 'machine learning python':")
for _, row in top5.iterrows():
    print(f"  [{row['score']:.3f}]  {row['course_title'][:55]}")

print("\nAll checks passed — Day 6 complete!")

Pickle verification:
  course_tfidf.pkl : TfidfVectorizer  vocab=5,000
  course_sim.pkl   : ndarray (3416, 3416)
  courses_df.pkl   : DataFrame (3416, 6)
  columns          : ['course_title', 'subject', 'level', 'rating', 'course_url', 'combined_text']

End-to-end test for 'machine learning python':
  [0.337]  Machine Learning with Python
  [0.306]  Applied Text Mining in Python
  [0.294]  Introduction to Reinforcement Learning in Python
  [0.288]  Applied Social Network Analysis in Python
  [0.282]  Introduction to Applied Machine Learning

All checks passed — Day 6 complete!


## 10. Methodology notes for BTP Report

### TF-IDF Content-Based Filtering

**TF-IDF (Term Frequency – Inverse Document Frequency)** converts text into numerical  
vectors where each dimension represents a word (or bigram).

**TF (Term Frequency):** how often a word appears in a course's combined_text.  
With `sublinear_tf=True`, we apply log(1 + tf) to reduce the dominance of very common words.

**IDF (Inverse Document Frequency):** log(N / df) where N = total courses, df = courses containing the word.  
Words that appear in every course ('the', 'and', 'course') get very low IDF — they don't help distinguish courses.  
Rare meaningful words ('bayesian', 'kubernetes', 'reinforcement') get high IDF.

**Cosine similarity** measures the angle between two TF-IDF vectors.  
A similarity of 1.0 means identical topics; 0.0 means completely unrelated.  
Unlike dot product, cosine similarity is length-independent — a 5-word title  
and a 20-word description can still score 1.0 if they discuss the same topic.

**Bigrams** (`ngram_range=(1,2)`) capture two-word phrases like 'machine learning',  
'data science', 'neural network' as single features — much more informative than  
treating each word independently.

**Why TF-IDF over collaborative filtering for courses?**  
The Coursera dataset has no user-course rating matrix — we don't know which users  
took which courses. TF-IDF requires only the course metadata (title, subject, level),  
making it ideal for content-based recommendations without user data.